# Data mirror
Copies the mounted competition data to /kaggle/working/data for local mirroring, then prints tape schemas.

In [ ]:
import os, shutil, time
src = None
for cand in ["/kaggle/input/competitions/memecoins-pump-or-dump", "/kaggle/input/memecoins-pump-or-dump"]:
    if os.path.isdir(cand):
        src = cand
        break
print("src:", src, flush=True)
out = "/kaggle/working/data"
os.makedirs(out, exist_ok=True)
t0 = time.time()
n = 0
tot = 0
for f in sorted(os.listdir(src)):
    p = os.path.join(src, f)
    if os.path.isfile(p):
        shutil.copy2(p, os.path.join(out, f))
        n += 1
        tot += os.path.getsize(p)
print(f"copied {n} files, {tot/1e9:.2f} GB in {time.time()-t0:.0f}s", flush=True)


In [ ]:

import pyarrow.parquet as pq
import glob
d = "/kaggle/working/data"
a = sorted(glob.glob(d + "/day_*_pumpswap_all_swaps.parquet"))[-1]
c = sorted(glob.glob(d + "/day_*_pumpfun_all_swaps.parquet"))[-1]
print("AMM cols:", pq.read_schema(a).names, flush=True)
print("CURVE cols:", pq.read_schema(c).names, flush=True)
import pandas as pd
adm = pd.read_parquet(a)
print("AMM base_token distinct:", adm["base_token"].nunique() if "base_token" in adm else "n/a", flush=True)
print("AMM rows:", len(adm), "mints:", adm["mint"].nunique(), flush=True)
print("fee_payer==signing_wallet frac:", float((adm["fee_payer"]==adm["signing_wallet"]).mean()) if "fee_payer" in adm else "n/a", flush=True)
csm = pd.read_parquet(c)
print("CURVE rows:", len(csm), "mints:", csm["mint"].nunique(), flush=True)
print("MIRROR DONE", flush=True)
